In [2]:
from kaggle_secrets import UserSecretsClient
secret_label = "HUGGINGFACE_HUB_TOKEN"
secret_value = UserSecretsClient().get_secret(secret_label)

In [3]:
import os
os.environ["HF_TOKEN"] = secret_value # Thay bằng token của bạn

In [4]:
from huggingface_hub import create_repo

# Thêm đoạn này trước khi chạy xử lý batch
try:
    create_repo(repo_id="rimine/ct-rate-medgemma-ready", repo_type="dataset", exist_ok=True)
    print("✅ Repo đã sẵn sàng!")
except Exception as e:
    print(f"Lỗi khi tạo repo: {e}")

✅ Repo đã sẵn sàng!


In [5]:
import os
# Số core CPU vật lý/logic khả dụng cho tiến trình của bạn
print("Số CPU Cores khả dụng:", os.cpu_count())

Số CPU Cores khả dụng: 4


In [ ]:
import os, sys, shutil, warnings, re
from concurrent.futures import ThreadPoolExecutor, as_completed
import nibabel as nib
import numpy as np
import cv2
import pandas as pd
from datasets import load_dataset, Dataset, Features, Value
from huggingface_hub import hf_hub_download, HfApi
from huggingface_hub.utils import disable_progress_bars
from tqdm import tqdm
import threading

warnings.filterwarnings("ignore")
disable_progress_bars()

# ==========================================
# 1. CẤU HÌNH THAM SỐ
# ==========================================
TARGET_SIZE  = None
MAX_SLICES   = 85
BATCH_SIZE   = 25
NUM_WORKERS  = 8

YOUR_HF_REPO      = "rimine/ct-rate-medgemma-ready"
TEMP_DOWNLOAD_DIR = "/kaggle/working/temp_ct_data"
NPZ_DIR           = "/kaggle/working/npz_output"
DONE_FLAG_FILE    = "/kaggle/working/processed_volumes.txt"

os.makedirs(TEMP_DOWNLOAD_DIR, exist_ok=True)
os.makedirs(NPZ_DIR, exist_ok=True)

_flag_lock = threading.Lock()
api = HfApi()

# ==========================================
# 2. CÁC HÀM XỬ LÝ
# ==========================================
INSTRUCTION = (
    "You are an expert radiologist specializing in computed tomography (CT). "
    "Analyze the provided contiguous block of CT slices carefully. "
    "Generate a detailed section for Findings and provide a final clinical "
    "Impression based on the visual evidence.\n\n"
)

def _to_hu(vol, slope, intercept):
    total = vol.shape[2]
    if total > MAX_SLICES:
        idx = [int(round(i / (MAX_SLICES - 1) * (total - 1))) for i in range(MAX_SLICES)]
        vol = vol[:, :, idx]
    return (vol * slope) + intercept

def _norm(s, lo, hi):
    s = np.clip(s, lo, hi).astype(np.float32)
    return np.round((s - lo) / (hi - lo) * 255).astype(np.uint8)

def _to_rgb(hu):
    wins = [(-1024, 1024), (-135, 215), (0, 80)]
    out = []
    for i in range(hu.shape[2]):
        sl  = hu[:, :, i]
        rgb = np.stack([_norm(sl, w[0], w[1]) for w in wins], axis=-1)
        if rgb.shape[:2] != (TARGET_SIZE, TARGET_SIZE):
            rgb = cv2.resize(rgb, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_AREA)
        out.append(rgb)
    return np.stack(out, axis=0)

def _hf_path(name):
    clean = name.replace(".nii.gz", "")
    m = re.match(r"(train_\d+)_([a-z])_\d+", clean)
    if m:
        fn = m.group(1)
        fl = f"{fn}_{m.group(2)}"
        return f"dataset/train/{fn}/{fl}/{name}"
    return f"dataset/train/{name}"

def process_single(row):
    name      = row["VolumeName"]
    slope     = float(row["RescaleSlope"])     if not pd.isna(row["RescaleSlope"])     else 1.0
    intercept = float(row["RescaleIntercept"]) if not pd.isna(row["RescaleIntercept"]) else -1024.0

    npz_filename = name.replace(".nii.gz", "") + ".npz"
    npz_local    = os.path.join(NPZ_DIR, npz_filename)
    npz_hf_path  = f"images/{npz_filename}"

    try:
        local = hf_hub_download(
            repo_id="ibrahimhamamci/CT-RATE",
            filename=_hf_path(name),
            repo_type="dataset",
            local_dir=TEMP_DOWNLOAD_DIR,
        )
        img  = nib.load(local)
        hu   = _to_hu(img.get_fdata(), slope, intercept)
        imgs = _to_rgb(hu)

        n = imgs.shape[0]
        if n < MAX_SLICES:
            pad  = np.zeros((MAX_SLICES - n, TARGET_SIZE, TARGET_SIZE, 3), dtype=np.uint8)
            imgs = np.concatenate([imgs, pad], axis=0)

        np.savez_compressed(npz_local, images=imgs)

        if os.path.exists(local):
            os.remove(local)

        return {
            "volume_name": name,
            "npz_path":    npz_hf_path,
            "prompt":      INSTRUCTION + "Please provide Findings and Impression for this CT volume scan.",
            "findings":    str(row.get("Findings_EN",  "")),
            "impressions":  str(row.get("Impressions_EN", "")),
        }
    except Exception as e:
        tqdm.write(f"  ⚠️  Skipping {name}: {e}")
        return None

# ==========================================
# 3. TẢI METADATA & MERGE
# ==========================================
print("🚀 Bước 1: Tải metadata từ CT-RATE...")
raw_reports  = load_dataset("ibrahimhamamci/CT-RATE", "reports",  split="train")
raw_metadata = load_dataset("ibrahimhamamci/CT-RATE", "metadata", split="train")

merged_df = pd.merge(
    pd.DataFrame(raw_reports),
    pd.DataFrame(raw_metadata)[["VolumeName", "RescaleSlope", "RescaleIntercept"]],
    on="VolumeName",
    how="inner",
)
print(f"📊 Tổng số mẫu: {len(merged_df)}")

# ==========================================
# 4. ĐỌC FLAG
# ==========================================
if os.path.exists(DONE_FLAG_FILE):
    with open(DONE_FLAG_FILE, "r") as f:
        done_set = set(line.strip() for line in f if line.strip())
    print(f"✅ Đã xử lý trước đó: {len(done_set)} volumes")
else:
    done_set = set()
    print("🆕 Bắt đầu từ đầu")

todo_df = merged_df[~merged_df["VolumeName"].isin(done_set)].reset_index(drop=True)
print(f"⏳ Còn lại: {len(todo_df)} volumes\n")

# ==========================================
# 5. XỬ LÝ MULTI-THREAD THEO BATCH → PUSH NGAY
# ==========================================
dataset_features = Features({
    "volume_name": Value("string"),
    "npz_path":    Value("string"),
    "prompt":      Value("string"),
    "findings":    Value("string"),
    "impressions":  Value("string"),
})

total          = len(todo_df)
n_batches      = (total + BATCH_SIZE - 1) // BATCH_SIZE

# Tính batch_idx bắt đầu dựa trên số shard đã có trên HF
# → tránh ghi đè shard cũ khi resume
existing_shards = [
    f for f in api.list_repo_files(YOUR_HF_REPO, repo_type="dataset")
    if f.startswith("data/train/shard-")
] if api.repo_exists(YOUR_HF_REPO, repo_type="dataset") else []
shard_offset = len(existing_shards)
print(f"📂 Số shard đã có trên HF: {shard_offset}\n")

print(f"⚡ Xử lý {total} volumes | batch={BATCH_SIZE} | workers={NUM_WORKERS}\n")

for batch_idx in range(n_batches):
    start    = batch_idx * BATCH_SIZE
    end      = min(start + BATCH_SIZE, total)
    batch_df = todo_df.iloc[start:end]

    print(f"{'='*55}")
    print(f"  Batch {batch_idx+1}/{n_batches}  |  volume {start+1}–{end}/{total}")
    print(f"{'='*55}")

    batch_records = []
    rows = [row for _, row in batch_df.iterrows()]

    # Bước 5a: xử lý song song → tạo file .npz
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = {executor.submit(process_single, row): row["VolumeName"] for row in rows}
        with tqdm(total=len(futures), desc=f"  Xử lý batch {batch_idx+1}", unit="vol") as pbar:
            for future in as_completed(futures):
                result = future.result()
                if result is not None:
                    batch_records.append(result)
                pbar.update(1)

    if not batch_records:
        print("  ⚠️  Batch rỗng, bỏ qua.\n")
        continue

    # Bước 5b: upload file .npz lên images/ (append — upload_folder không xóa file cũ)
    print(f"  📦 Uploading {len(batch_records)} file .npz...")
    api.upload_folder(
        repo_id=YOUR_HF_REPO,
        repo_type="dataset",
        folder_path=NPZ_DIR,
        path_in_repo="images",
        delete_patterns=None,       # KHÔNG xóa file cũ → append
    )

    # Bước 5c: lưu metadata batch thành 1 shard parquet riêng → upload
    # Mỗi batch = 1 file shard độc lập, không đụng đến shard cũ
    shard_idx       = shard_offset + batch_idx
    shard_filename  = f"shard-{shard_idx:05d}-of-99999.parquet"
    shard_local     = f"/kaggle/working/{shard_filename}"
    shard_hf_path   = f"data/train/{shard_filename}"

    print(f"  📤 Uploading metadata shard: {shard_hf_path}")
    Dataset.from_list(batch_records, features=dataset_features).to_parquet(shard_local)
    api.upload_file(
        repo_id=YOUR_HF_REPO,
        repo_type="dataset",
        path_or_fileobj=shard_local,
        path_in_repo=shard_hf_path,     # file mới hoàn toàn → không overwrite shard cũ
    )
    os.remove(shard_local)

    # Bước 5d: ghi flag + dọn .npz local
    with _flag_lock:
        with open(DONE_FLAG_FILE, "a") as f:
            for rec in batch_records:
                f.write(rec["volume_name"] + "\n")
        done_set.update(rec["volume_name"] for rec in batch_records)

    npz_local_paths = [os.path.join(NPZ_DIR, os.path.basename(r["npz_path"])) for r in batch_records]
    for p in npz_local_paths:
        if os.path.exists(p):
            os.remove(p)

    print(f"  ✅ Xong batch {batch_idx+1} — tổng: {len(done_set)}/{len(merged_df)}\n")

# ==========================================
# 6. DỌN DẸP
# ==========================================
if os.path.exists(TEMP_DOWNLOAD_DIR):
    shutil.rmtree(TEMP_DOWNLOAD_DIR)
if os.path.exists(NPZ_DIR):
    shutil.rmtree(NPZ_DIR)

print("\n🎉 Hoàn tất!")
print(f"   Ảnh .npz : {YOUR_HF_REPO}/images/")
print(f"   Metadata : {YOUR_HF_REPO}/data/train/shard-NNNNN-of-99999.parquet")
print(f"   Flag file: {DONE_FLAG_FILE}")

🚀 Bước 1: Tải metadata từ CT-RATE...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

...

  Xử lý batch 14: 100%|██████████| 25/25 [04:33<00:00, 10.96s/vol]


  📦 Uploading 25 file .npz...
  📤 Uploading metadata shard: data/train/shard-00058-of-99999.parquet


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

  ✅ Xong batch 14 — tổng: 1475/47149

  Batch 15/1841  |  volume 351–375/46024


  Xử lý batch 15:  72%|███████▏  | 18/25 [03:14<01:00,  8.67s/vol]